# DBSCAN vs HDBSCAN — 공식 데모(이론)부터 우리 운전자 데이터까지

| 단계 | 내용 |
|---|---|
| **§1 이론** | scikit-learn **공식 예제 코드 원본**을 그대로 실행 — DBSCAN/HDBSCAN이 언제 갈라지는지 |
| **§2 커스터마이징** | MASIL이 표준 알고리즘에서 무엇을 바꿨고 왜 바꿨는지 — 코드로 확인 |
| **§3~5 우리 데이터** | 실제 운전자 데이터에서 두 방식이 어떻게 표현되는지 + 차이가 나타나는 조건 + eps 측정 절차 |

**사용법 (Colab)**: `런타임 → 모두 실행`. 첫 셀이 GitHub에서 데이터를 자동으로 받아옵니다.

공식 문서 (이론 원문):
- DBSCAN 개념: https://scikit-learn.org/stable/modules/clustering.html#dbscan
- HDBSCAN 개념: https://scikit-learn.org/stable/modules/clustering.html#hdbscan
- 공식 데모 — DBSCAN: https://scikit-learn.org/stable/auto_examples/cluster/plot_dbscan.html
- 공식 데모 — HDBSCAN: https://scikit-learn.org/stable/auto_examples/cluster/plot_hdbscan.html
- 알고리즘 총비교: https://scikit-learn.org/stable/auto_examples/cluster/plot_cluster_comparison.html


In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("seniorcareservice"):
        !git clone --branch claude/gaip-dashboard-refine --depth 1 https://github.com/summit1123/seniorcareservice.git
    ROOT = "seniorcareservice"
else:
    ROOT = ".."  # 로컬에서 notebooks/ 폴더 기준
sys.path.insert(0, ROOT)
print("데이터 위치:", ROOT + "/data/fixtures/gaip_visit_events.csv")

## §1-a 공식 데모: DBSCAN

아래 두 셀은 scikit-learn 공식 예제 **"Demo of DBSCAN clustering algorithm"** 원본 코드입니다
(출처·라이선스는 셀 상단 주석). 밀도가 비슷한 무리 3개를 eps=0.3으로 잡는 기본 동작과,
정답 라벨 없이도 쓸 수 있는 평가지표(Silhouette)를 보여줍니다 — 실데이터 검증 때 우리가
라벨-프리 지표를 쓰는 것과 같은 맥락입니다.

In [ ]:
# 출처: scikit-learn 공식 예제 "Demo of DBSCAN clustering algorithm" (원본)
# https://scikit-learn.org/stable/auto_examples/cluster/plot_dbscan.html
# Authors: The scikit-learn developers | SPDX-License-Identifier: BSD-3-Clause

from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

centers = [[1, 1], [-1, -1], [1, -1]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=0.4, random_state=0
)

X = StandardScaler().fit_transform(X)

import numpy as np

from sklearn import metrics
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.3, min_samples=10).fit(X)
labels = db.labels_

n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)
print(f"Homogeneity: {metrics.homogeneity_score(labels_true, labels):.3f}")
print(f"Completeness: {metrics.completeness_score(labels_true, labels):.3f}")
print(f"V-measure: {metrics.v_measure_score(labels_true, labels):.3f}")
print(f"Adjusted Rand Index: {metrics.adjusted_rand_score(labels_true, labels):.3f}")
print(
    "Adjusted Mutual Information:"
    f" {metrics.adjusted_mutual_info_score(labels_true, labels):.3f}"
)
print(f"Silhouette Coefficient: {metrics.silhouette_score(X, labels):.3f}")

In [ ]:
# (이어서 원본의 결과 플롯 — 큰 점 = core sample, 작은 점 = non-core, 검정 = 노이즈)
import matplotlib.pyplot as plt

unique_labels = set(labels)
core_samples_mask = np.zeros_like(labels, dtype=bool)
core_samples_mask[db.core_sample_indices_] = True

colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
for k, col in zip(unique_labels, colors):
    if k == -1:
        col = [0, 0, 0, 1]

    class_member_mask = labels == k

    xy = X[class_member_mask & core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], "o", markerfacecolor=tuple(col),
             markeredgecolor="k", markersize=14)

    xy = X[class_member_mask & ~core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], "o", markerfacecolor=tuple(col),
             markeredgecolor="k", markersize=6)

plt.title(f"Estimated number of clusters: {n_clusters_}")
plt.show()

## §1-b 공식 데모: HDBSCAN

이하 셀들은 공식 예제 **"Demo of HDBSCAN clustering algorithm"** 원본 코드를 섹션별로 묶은 것입니다.
우리 논쟁과의 연결점을 섹션마다 표시했습니다.

In [ ]:
# 출처: scikit-learn 공식 예제 "Demo of HDBSCAN clustering algorithm" (원본)
# https://scikit-learn.org/stable/auto_examples/cluster/plot_hdbscan.html
# Authors: The scikit-learn developers | SPDX-License-Identifier: BSD-3-Clause

import matplotlib.pyplot as plt
import numpy as np

from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.datasets import make_blobs


def plot(X, labels, probabilities=None, parameters=None, ground_truth=False, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    labels = labels if labels is not None else np.ones(X.shape[0])
    probabilities = probabilities if probabilities is not None else np.ones(X.shape[0])
    # Black removed and is used for noise instead.
    unique_labels = set(labels)
    colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
    # The probability of a point belonging to its labeled cluster determines
    # the size of its marker
    proba_map = {idx: probabilities[idx] for idx in range(len(labels))}
    for k, col in zip(unique_labels, colors):
        if k == -1:
            # Black used for noise.
            col = [0, 0, 0, 1]

        class_index = (labels == k).nonzero()[0]
        for ci in class_index:
            ax.plot(
                X[ci, 0],
                X[ci, 1],
                "x" if k == -1 else "o",
                markerfacecolor=tuple(col),
                markeredgecolor="k",
                markersize=4 if k == -1 else 1 + 5 * proba_map[ci],
            )
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    preamble = "True" if ground_truth else "Estimated"
    title = f"{preamble} number of clusters: {n_clusters_}"
    if parameters is not None:
        parameters_str = ", ".join(f"{k}={v}" for k, v in parameters.items())
        title += f" | {parameters_str}"
    ax.set_title(title)
    plt.tight_layout()

### 스케일 불변성 — "eps는 데이터의 축척에 민감하다"

같은 데이터를 0.5배/3배로 늘렸을 뿐인데 같은 eps=0.3의 결과가 완전히 달라집니다.
HDBSCAN은 축척과 무관하게 같은 답을 냅니다.
**(우리 논쟁 연결: "왜 도 단위 0.012가 아니라 미터 haversine으로 통일했나"의 이론적 배경)**

In [ ]:
# (원본 § Scale Invariance)
centers = [[1, 1], [-1, -1], [1.5, -1.5]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=[0.4, 0.1, 0.75], random_state=0
)

fig, axes = plt.subplots(3, 1, figsize=(10, 12))
dbs = DBSCAN(eps=0.3)
for idx, scale in enumerate([1, 0.5, 3]):
    dbs.fit(X * scale)
    plot(X * scale, dbs.labels_, parameters={"scale": scale, "eps": 0.3}, ax=axes[idx])

fig, axes = plt.subplots(3, 1, figsize=(10, 12))
hdb = HDBSCAN(copy=True)
for idx, scale in enumerate([1, 0.5, 3]):
    hdb.fit(X * scale)
    plot(
        X * scale,
        hdb.labels_,
        hdb.probabilities_,
        ax=axes[idx],
        parameters={"scale": scale},
    )

### 멀티스케일(밀도 혼합) — 공식 데모의 핵심 장면

촘촘한 무리 2개 + 성긴 무리 2개가 **한 판에** 섞인 데이터입니다. 공식 문서의 결론:
eps가 크면(0.7) 촘촘한 두 무리가 병합되고, 작으면(0.3) 성긴 무리가 파편화됩니다 —
**어떤 eps를 골라도 동시에 만족 못 함.** HDBSCAN은 무리별 스케일을 따로 잡아 넷 다 복원합니다.

**(우리 논쟁 연결: 이 장면의 전제는 "한 판에 밀도 혼합". 우리는 사람별로 따로 군집화하므로
이 전제가 약해지고 — §3에서 보듯 평상시 데이터에선 둘의 차이가 사라집니다.)**

In [ ]:
# (원본 § Multi-Scale Clustering)
centers = [[-0.85, -0.85], [-0.85, 0.85], [3, 3], [3, -3]]
X, labels_true = make_blobs(
    n_samples=750, centers=centers, cluster_std=[0.2, 0.35, 1.35, 1.35], random_state=0
)
plot(X, labels=labels_true, ground_truth=True)

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
params = {"eps": 0.7}
dbs = DBSCAN(**params).fit(X)
plot(X, dbs.labels_, parameters=params, ax=axes[0])
params = {"eps": 0.3}
dbs = DBSCAN(**params).fit(X)
plot(X, dbs.labels_, parameters=params, ax=axes[1])

hdb = HDBSCAN(copy=True).fit(X)
plot(X, hdb.labels_, hdb.probabilities_)

### 하이퍼파라미터 민감도 — "HDBSCAN도 손잡이가 있다"

공식 문서 스스로 말합니다: HDBSCAN은 eps에서 자유롭지만 `min_cluster_size`와 `min_samples`가
결과를 좌우한다고. 아래에서 min_cluster_size 3/5/25, min_samples 3/5/25에 따라 결과가
어떻게 변하는지 보세요. **(우리 논쟁 연결: "파라미터는 사라지지 않고 이동한다" — 우리 실측에서
3/2 → 5/3 변경만으로 파편화율이 47% → 12%로 출렁였던 이유)**

In [ ]:
# (원본 § Hyperparameter Robustness — min_cluster_size / min_samples)
PARAM = ({"min_cluster_size": 5}, {"min_cluster_size": 3}, {"min_cluster_size": 25})
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
for i, param in enumerate(PARAM):
    hdb = HDBSCAN(copy=True, **param).fit(X)
    labels = hdb.labels_

    plot(X, labels, hdb.probabilities_, param, ax=axes[i])

PARAM = (
    {"min_cluster_size": 20, "min_samples": 5},
    {"min_cluster_size": 20, "min_samples": 3},
    {"min_cluster_size": 20, "min_samples": 25},
)
fig, axes = plt.subplots(3, 1, figsize=(10, 12))
for i, param in enumerate(PARAM):
    hdb = HDBSCAN(copy=True, **param).fit(X)
    labels = hdb.labels_

    plot(X, labels, hdb.probabilities_, param, ax=axes[i])

## §2 MASIL 커스터마이징 — 표준에서 무엇을 바꿨나

| # | 표준 (sklearn) | MASIL (`src/gaip_simulation/clustering.py`) | 왜 |
|---|---|---|---|
| 1 | 유클리드 거리 (좌표 단위) | **haversine 미터 거리** | §1-b 스케일 불변성에서 봤듯 eps는 축척에 민감 — 위경도 도 단위는 방향에 따라 길이가 달라 왜곡됨 |
| 2 | `min_samples` = 이웃 **점 개수** | **"서로 다른 방문일 3일 이상"** (`min_distinct_days`) | 하루에 10번 정차한 곳(행사·이사)이 거점으로 오인되는 것 방지 — 상품 규칙을 알고리즘 코어 판정에 내장 |
| 3 | 군집 = 최종 출력 | 군집 위에 **상품 존 층**: 중심 + max(코어 500m, 방문거리 P90, 상한 2,000m) 원의 **합집합** | 판정(생활권 안/밖)은 원 합집합 기준 — 군집 모양이 아니라 반복 방문 여부가 상품의 관심사 |
| 4 | 파라미터 = 사용자가 임의 지정 | eps를 **지역 풀링 k-거리 실측으로 도출**하는 절차 + 9지표 채점표 | "왜 이 숫자냐"를 상수 방어가 아니라 측정 절차로 답하기 위해 |

아래 셀이 2번 커스터마이징의 효과를 직접 보여줍니다: 실제 운전자 데이터에
**"하루에만 10번 정차한 가짜 지점"**을 심은 뒤, 표준 DBSCAN과 우리 변형을 비교합니다.

In [ ]:
import csv, math
from collections import defaultdict
from src.gaip_simulation.clustering import dbscan_distinct_days, haversine_m, percentile_nearest_rank

CSV_PATH = ROOT + "/data/fixtures/gaip_visit_events.csv"
CORE_M, CAP_M = 500.0, 2000.0
COLORS = ["#1D9E75", "#378ADD", "#EF9F27", "#D4537E", "#7F77DD", "#639922", "#D85A30", "#0F6E56"]

with open(CSV_PATH, encoding="utf-8") as f:
    ALL_ROWS = list(csv.DictReader(f))
for r in ALL_ROWS:
    r["latitude"] = float(r["latitude"]); r["longitude"] = float(r["longitude"])

def load_driver(driver_id):
    rows = [r for r in ALL_ROWS if r["driver_id"] == driver_id]
    fit = [r for r in rows if r["period_role"] == "baseline"]
    ev = [r for r in rows if r["period_role"] != "baseline"]
    return fit, ev, rows[0]["environment_id"], rows[0]["designed_type"]

fit, _, _, _ = load_driver("gaip-003")
anchor_lat = sum(e["latitude"] for e in fit) / len(fit)
anchor_lon = sum(e["longitude"] for e in fit) / len(fit)

# 가짜 지점: 실제 거점들에서 2km 떨어진 곳, 하루(2025-11-15)에만 10번 정차
import random
rr = random.Random(3)
fake_lat = anchor_lat + 2000 / 111_320.0
one_day_burst = [{"latitude": fake_lat + rr.gauss(0, 25) / 111_320.0,
                  "longitude": anchor_lon + rr.gauss(0, 25) / 88_000.0,
                  "visit_date": "2025-11-15"} for _ in range(10)]
mixed = [{"latitude": e["latitude"], "longitude": e["longitude"], "visit_date": e["visit_date"]}
         for e in fit] + one_day_burst

# 표준 DBSCAN (min_samples=점 3개) vs MASIL (서로 다른 3일)
from sklearn.cluster import DBSCAN as SkDBSCAN
xy = np.array([((e["longitude"] - anchor_lon) * 111_320.0 * math.cos(math.radians(anchor_lat)),
                (e["latitude"] - anchor_lat) * 111_320.0) for e in mixed])
vanilla = SkDBSCAN(eps=260, min_samples=3).fit_predict(xy)
masil = dbscan_distinct_days(mixed, eps_m=260, min_distinct_days=3)["labels"]

def burst_result(labels):
    burst_labels = labels[-10:]
    return "거점으로 인정" if max(burst_labels) >= 0 else "노이즈로 기각"

print("하루 10번 정차한 가짜 지점의 운명:")
print(f"  표준 DBSCAN(min_samples=3)      -> {burst_result(list(vanilla))}")
print(f"  MASIL(서로 다른 3일 규칙)        -> {burst_result(masil)}")
print()
print("같은 eps, 같은 데이터 — '점 3개'와 '3일'의 차이가 하루 몰림 오인을 가릅니다.")

## §3 우리 데이터 — 실제 운전자에서 두 방식의 표현

이제 실제 운전자 한 명에게 4가지 설정을 나란히 적용합니다.

**그림 읽는 법**: 점 = 기준선(첫 2개월) 방문 · 색 = 군집 · 회색 × = 노이즈 ·
점선 원 = 생활권 원(반경 = max(500m, P90), 상한 2km) · **최종 생활권 = 원 합집합** ·
`eval coverage` = 이후 12개월 방문이 그 합집합에 들어온 비율

In [ ]:
def offsets(events, anchor):
    lat0, lon0 = anchor
    return [((e["longitude"] - lon0) * 111_320.0 * math.cos(math.radians(lat0)),
             (e["latitude"] - lat0) * 111_320.0) for e in events]

def zone_clusters(events, labels):
    groups = defaultdict(list)
    for e, l in zip(events, labels):
        if l >= 0:
            groups[l].append(e)
    out = []
    for l, evs in sorted(groups.items()):
        lat = sum(e["latitude"] for e in evs) / len(evs)
        lon = sum(e["longitude"] for e in evs) / len(evs)
        d = [haversine_m(e["latitude"], e["longitude"], lat, lon) for e in evs]
        p90 = percentile_nearest_rank(d, 0.90)
        out.append({"lat": lat, "lon": lon, "r": max(CORE_M, min(p90, CAP_M))})
    return out

def coverage(eval_events, clusters):
    if not clusters or not eval_events:
        return 0.0
    inz = sum(1 for e in eval_events
              if any(haversine_m(e["latitude"], e["longitude"], c["lat"], c["lon"]) <= c["r"]
                     for c in clusters))
    return inz / len(eval_events) * 100

def run_hdbscan(events, anchor, mcs, ms):
    from sklearn.cluster import HDBSCAN
    xy = offsets(events, anchor)
    h = HDBSCAN(min_cluster_size=mcs, min_samples=ms, allow_single_cluster=True, copy=True)
    return h.fit_predict(np.array(xy)).tolist()

def compare(driver_id, configs=None):
    if configs is None:
        configs = [
            ("DBSCAN eps=260m + 3 days", "dbscan", (260, 3)),
            ("DBSCAN eps=1100m + 3 days", "dbscan", (1100, 3)),
            ("HDBSCAN mcs=3, ms=2", "hdbscan", (3, 2)),
            ("HDBSCAN mcs=5, ms=3", "hdbscan", (5, 3)),
        ]
    fit, ev, env, dtype = load_driver(driver_id)
    anchor = (sum(e["latitude"] for e in fit) / len(fit),
              sum(e["longitude"] for e in fit) / len(fit))
    n = len(configs)
    nrows = (n + 1) // 2
    fig, axes = plt.subplots(nrows, 2, figsize=(13, 6 * nrows))
    axes = axes.flat if n > 1 else [axes]
    for ax, (title, algo, params) in zip(axes, configs):
        if algo == "dbscan":
            labels = dbscan_distinct_days(fit, eps_m=params[0], min_distinct_days=params[1])["labels"]
        else:
            labels = run_hdbscan(fit, anchor, *params)
        clusters = zone_clusters(fit, labels)
        cov = coverage(ev, clusters)
        for (x, y), l in zip(offsets(fit, anchor), labels):
            if l < 0:
                ax.scatter(x, y, marker="x", c="#999999", s=42, zorder=3)
            else:
                ax.scatter(x, y, c=COLORS[l % len(COLORS)], s=30, zorder=3, edgecolors="none")
        for i, c in enumerate(clusters):
            cx = (c["lon"] - anchor[1]) * 111_320.0 * math.cos(math.radians(anchor[0]))
            cy = (c["lat"] - anchor[0]) * 111_320.0
            ax.add_patch(plt.Circle((cx, cy), c["r"], fill=True, alpha=0.10,
                                    color=COLORS[i % len(COLORS)], zorder=1))
            ax.add_patch(plt.Circle((cx, cy), c["r"], fill=False, linestyle="--", linewidth=1.4,
                                    color=COLORS[i % len(COLORS)], zorder=2))
        noise = sum(1 for l in labels if l < 0)
        ax.set_title(f"{title}\nclusters={len(clusters)}  noise={noise}  eval coverage={cov:.1f}%",
                     fontsize=11)
        ax.set_aspect("equal"); ax.grid(alpha=0.25)
        ax.set_xlabel("east (m)"); ax.set_ylabel("north (m)")
    fig.suptitle(f"driver {driver_id} ({env}, {dtype}) - baseline visits, zone = union of circles",
                 fontsize=13)
    fig.tight_layout()
    plt.show()

DRIVER_ID = "gaip-123"   # 광역 · 다생활권형 — 바꿔서 실행해 보세요
compare(DRIVER_ID)

In [ ]:
# 유형 x 환경 대표 운전자 목록 — DRIVER_ID를 바꿔가며 확인
reps = {}
for r in ALL_ROWS:
    key = (r["environment_id"], r["designed_type"])
    reps.setdefault(key, r["driver_id"])
print(f"{'환경':24s} {'유형':26s} 대표 운전자")
for (env, t), d in sorted(reps.items()):
    print(f"{env:24s} {t:26s} {d}")

### 여기서 확인하게 되는 것

1. **원의 개수는 설정마다 다른데, 원 합집합(최종 생활권)은 거의 같습니다.** 파편화는 상품 층에서 무해.
2. 눈에 보이는 유일한 손실은 **큰 eps(1100)의 병합** — 남쪽 거점들이 합쳐지고 버퍼 상한에 걸림.
   §1-b 멀티스케일 데모의 "eps가 크면 병합" 그대로입니다.
3. HDBSCAN 5/3 ≈ DBSCAN 260. **평상시 데이터에서 둘의 차이는 사실상 없습니다.**

§1의 이론적 차이가 왜 안 보일까요? 우리 합성 데이터의 주차 산포(수십 m)가
eps(260m)보다 한참 작아서입니다. 그러면 차이가 *언제* 나타나는지 — 다음 셀이 그 답입니다.

## §4 차이가 나타나는 조건 — 산포 스트레스 테스트

주차 산포(GPS 흩어짐)를 인위적으로 σ만큼 키우면서 광역 60명을 다시 돌립니다.
산포가 eps(260m)에 접근/초과하는 순간 두 알고리즘의 본성이 갈라집니다:
고정 자(eps)는 체인이 끊겨 부서지고, HDBSCAN은 스케일을 늘려 버팁니다.

In [ ]:
import random

def jitter(events, sigma, rng):
    out = []
    for e in events:
        out.append({"latitude": e["latitude"] + rng.gauss(0, sigma) / 111_320.0,
                    "longitude": e["longitude"] + rng.gauss(0, sigma)
                    / (111_320.0 * math.cos(math.radians(e["latitude"]))),
                    "visit_date": e["visit_date"]})
    return out

wide = defaultdict(lambda: {"fit": [], "ev": []})
for r in ALL_ROWS:
    if r["environment_id"] != "wide_low_density":
        continue
    e = {"latitude": r["latitude"], "longitude": r["longitude"], "visit_date": r["visit_date"]}
    wide[r["driver_id"]]["fit" if r["period_role"] == "baseline" else "ev"].append(e)

SIGMA_LIST = [0, 100, 200, 400]   # 산포를 키워가며 — 400m은 eps 260을 넘는 구간
print(f"{'산포σ':>6s} | {'알고리즘':16s} | {'존실패':>4s} | {'평균거점':>6s} | {'노이즈%':>6s} | {'커버리지%':>7s}")
for sigma in SIGMA_LIST:
    for name, fn in [("DBSCAN e260/3일",
                      lambda f: dbscan_distinct_days(f, eps_m=260, min_distinct_days=3)["labels"]),
                     ("HDBSCAN 5/3",
                      lambda f: run_hdbscan(f, (sum(e['latitude'] for e in f) / len(f),
                                                sum(e['longitude'] for e in f) / len(f)), 5, 3))]:
        rng2 = random.Random(42)
        zero = 0; hubs = []; noise = 0; pts = 0; covs = []
        for d, dd in wide.items():
            fitj = jitter(dd["fit"], sigma, rng2)
            evj = jitter(dd["ev"], sigma, rng2)
            labels = fn(fitj)
            zs = zone_clusters(fitj, labels)
            if not zs:
                zero += 1
            hubs.append(len(zs)); noise += sum(1 for l in labels if l < 0); pts += len(labels)
            covs.append(coverage(evj, zs))
        print(f"{sigma:5d}m | {name:16s} | {zero:3d}명 | {sum(hubs)/len(hubs):6.1f} "
              f"| {noise/pts*100:5.1f} | {sum(covs)/len(covs):6.1f}")

### 스트레스 테스트가 말해주는 것

- **σ ≤ 100m (현 합성 수준)**: 두 알고리즘 사실상 동일 — "차이가 안 느껴지는" 게 정상인 구간.
- **σ = 400m (eps 초과)**: DBSCAN은 거점이 4조각으로 갈라지고 방문 20%를 버리며 존 실패자가 등장.
  HDBSCAN은 스케일을 늘려 거점 수를 유지하고 커버리지에서 역전.

→ **두 알고리즘의 차이는 "실측 주차 산포가 eps에 접근하는 지역"에서만 존재합니다.**
그래서 아래 k-거리 산포 측정이 단순 파라미터 튜닝이 아니라 **알고리즘 선택 스위치**가 됩니다:
산포 ≪ eps → 설명 가능한 DBSCAN으로 충분 / 산포 ≳ eps → eps 상향 또는 HDBSCAN 검토.

## §5 eps를 "정하지 않고 재는" 절차 — k-거리 산포 측정

같은 장소를 **다른 날** 재방문했을 때 좌표가 서로 얼마나 떨어져 찍히는지를 지역별로 모아
백분위를 보면, eps를 사람이 정하지 않고 데이터에서 도출할 수 있습니다.
실데이터 파일럿에서 이 절차를 그대로 쓰면 됩니다.

In [ ]:
by_env_driver = defaultdict(lambda: defaultdict(list))
for r in ALL_ROWS:
    if r["period_role"] == "baseline":
        by_env_driver[r["environment_id"]][r["driver_id"]].append(
            (r["latitude"], r["longitude"], r["visit_date"]))

def pct(xs, q):
    xs = sorted(xs)
    return xs[max(0, min(len(xs) - 1, int(len(xs) * q)))]

print(f"{'환경':24s} {'P50':>6s} {'P90':>6s} {'P95':>6s}   시사 eps")
for env, drivers in by_env_driver.items():
    pooled = []
    for d, pts in drivers.items():
        for i, (la, lo, dt) in enumerate(pts):
            ds = sorted(haversine_m(la, lo, la2, lo2)
                        for j, (la2, lo2, dt2) in enumerate(pts) if j != i and dt2 != dt)
            if len(ds) >= 2:
                pooled.append(ds[1])
    p50, p90, p95 = pct(pooled, 0.50), pct(pooled, 0.90), pct(pooled, 0.95)
    print(f"{env:24s} {p50:5.0f}m {p90:5.0f}m {p95:5.0f}m   ~{math.ceil(p95 / 10) * 10}m")
print()
print("합성 데이터의 산포(수십 m)는 실제보다 훨씬 촘촘합니다 — 이 표의 값 자체가 아니라")
print("'절차가 지역별 산포 차이를 복원한다'는 사실이 검증 포인트입니다.")

## 정리 — 이 노트북이 보여준 것

1. **이론(§1, 공식 데모 원본)**: 고정 eps의 실패(축척 민감·밀도 혼합)와 HDBSCAN의 존재 이유는
   실재한다 — 동시에 공식 문서 스스로 HDBSCAN의 min_cluster_size/min_samples 민감도를 인정한다.
2. **커스터마이징(§2)**: MASIL은 표준 DBSCAN에 haversine 미터 + **서로 다른 3일 규칙** +
   상품 존 층(원 합집합) + 채점표를 얹었다. "점 3개"가 아니라 "3일"이라서 하루 몰림이 거점으로
   오인되지 않는다.
3. **우리 데이터(§3~4)**: 평상시 산포에선 두 알고리즘이 같은 생활권을 만든다. 차이는 산포가
   eps에 접근할 때만 나타나며, 그 경계는 k-거리 측정(§5)으로 잰다.

| | A안: DBSCAN(3일 내장) | B안: HDBSCAN + 3일 후필터 |
|---|---|---|
| 판정 규칙 문장화 | "반경 ε 안, 서로 다른 3일 방문 = 거점" ✅ | 계층 안정성 — 문장화 어려움 ❌ |
| 공간 스케일 출처 | 지역 풀링 k-거리 실측 | 개인 데이터 자동 (개인당 점 ~100개) |
| 남는 검증 과제 | eps 컷 백분위 | min_samples 민감도 |

**현재 결론**: 운영·발표는 A안(설명 가능·구현 완료) 유지. B안은 실데이터 파일럿에서 같은 채점표
(커버리지·노이즈·거점수·기간 안정성 IoU)로 베이크오프하고, 실측 산포가 eps에 접근하는 지역이
확인되면 그 지역부터 eps 상향 또는 B안 전환을 검토한다.